In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import sum, expr, col, when, lit, regexp_extract, percentile_approx, regexp_replace, expr, count, to_timestamp, unix_timestamp, mean
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
import pyspark.sql.types as T
from pyspark.sql.window import Window

from pyspark.storagelevel import StorageLevel
from graphframes import GraphFrame

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import holidays

Using a configurable base data directory for train/val/test that are parquet files partitioned by year and month (can use the base of "dbfs:/student-groups/Group_3_1/mvp/data-separated" as an example starter data, with the train, val, test directories within), please create a series of models for Random Forest. The dataframes are already assembled into "features", "label", "YEAR" (2015-2019), and "MONTH" (1-12). The data is of 2015-2019, where 2015-2017 data will be train, 2018 will be val, and 2019 will be test. The time series Cross Validation Strategy should be a sliding window of 4 quarters train and 1 quarter val from 2015 - 2017, resulting in 8 sliding windows. The resultant and checkpoint directories can be within `dbfs:/student-groups/Group_3_1/mvp/stronger_models`. Please use mllib instead of sklearn and utilize dbutils instead of os if possible. Please output training and test results with priority to unweighted F2 score and PR-AUC

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.sql.functions import concat_ws
import numpy as np

# Configurable base directory
base_dir = "dbfs:/student-groups/Group_3_1/mvp/data_separated"
result_dir = "dbfs:/student-groups/Group_3_1/mvp/random_forest/results"
checkpoint_dir = "dbfs:/student-groups/Group_3_1/mvp/random_forest/checkpoints"
grid_search_checkpoints_dir = "dbfs:/student-groups/Group_3_1/mvp/random_forest/grid_search/checkpoints"
grid_search_results_dir = "dbfs:/student-groups/Group_3_1/mvp/random_forest/grid_search/results"

# Helper to load data for given years and months
def load_data(years, months=None):
    paths = []
    for y in years:
        if months:
            for m in range(1,13):
                paths.append(f"{base_dir}/train/YEAR={y}/MONTH={m}")
        else:
            paths.append(f"{base_dir}/train/YEAR={y}")
    return spark.read.parquet(*paths)

# Helper to get all months in a year
def get_months():
    return [str(m) for m in range(1, 13)]

# Sliding window setup: 2015-2018, 6 quarters train, 2 quarter val, 6 windows
windows = []
for start_q in range(0, 5):
    train_start_month = 1 + start_q * 6
    train_end_month = train_start_month + 17
    val_start_month = train_end_month + 1
    val_end_month = val_start_month + 5

    train_years = []
    train_months = []
    val_years = []
    val_months = []

    for i in range(train_start_month, train_end_month + 1):
        y = 2015 + (i - 1) // 12
        m = ((i - 1) % 12) + 1
        train_years.append(y)
        train_months.append(str(m).zfill(2))
    for i in range(val_start_month, val_end_month + 1):
        y = 2015 + (i - 1) // 12
        m = ((i - 1) % 12) + 1
        val_years.append(y)
        val_months.append(str(m).zfill(2))
    windows.append({
        "train": {"years": train_years, "months": train_months},
        "val": {"years": val_years, "months": val_months}
    })

In [0]:
windows

In [0]:
len(windows)

In [0]:
# F2 score evaluator
def f2_score(pred_df):
    tp = pred_df.filter((col('label') == 1) & (col('prediction') == 1)).count()
    fp = pred_df.filter((col('label') == 0) & (col('prediction') == 1)).count()
    fn = pred_df.filter((col('label') == 1) & (col('prediction') == 0)).count()
    tn = pred_df.filter((col('label') == 0) & (col('prediction') == 0)).count()
    if tp + fp == 0 or tp + fn == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if precision + recall == 0:
        return 0.0
    return tp, fp, fn, tn, precision, recall, 5 * (precision * recall) / (4 * precision + recall)

# PR-AUC evaluator
pr_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")

### Random Forest

In [0]:
# Model training and evaluation
results = []
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=200, maxDepth=16, seed=42)

for idx, win in enumerate(windows):
    print(f"Window {idx}")
    print("Loading data")
    train_df = load_data(win["train"]["years"], win["train"]["months"])
    val_df = load_data(win["val"]["years"], win["val"]["months"])

    print("Training model")
    model = rf.fit(train_df)
    val_pred = model.transform(val_df)
    train_pred = model.transform(train_df)

    print("Evaluating model")
    tp_val, fp_val, fn_val, tn_val, precision_val, recall_val, f2_val = f2_score(val_pred)
    pr_auc_val = pr_evaluator.evaluate(val_pred)
    tp_train, fp_train, fn_train, tn_train, precision_train, recall_train, f2_train = f2_score(train_pred)
    pr_auc_train = pr_evaluator.evaluate(train_pred)

    print("Evaluating model done")
    print("Scores:")
    print("Val:")
    print(f"tp_val: {tp_val}")
    print(f"fp_val: {fp_val}")
    print(f"fn_val: {fn_val}")
    print(f"tn_val: {tn_val}")
    print(f"precision_val: {precision_val}")
    print(f"recall_val: {recall_val}")
    print(f"f2_val: {f2_val}")
    print(f"pr_auc_val: {pr_auc_val} \n")
    print("Train:")
    print(f"tp_train: {tp_train}")
    print(f"fp_train: {fp_train}")
    print(f"fn_train: {fn_train}")
    print(f"tn_train: {tn_train}")
    print(f"precision_train: {precision_train}")
    print(f"recall_train: {recall_train}")
    print(f"f2_train: {f2_train}")
    print(f"pr_auc_train: {pr_auc_train}")
    results.append({
        "window": idx,
        "f2_val": f2_val,
        "pr_auc_val": pr_auc_val,
        "f2_train": f2_train,
        "pr_auc_train": pr_auc_train
    })

    model_path = f"{checkpoint_dir}/testnonLeakage/rf_window_{idx}"
    model.write().overwrite().save(model_path)
    print("Model iteration saved\n\n")


In [0]:
display(dbutils.fs.ls(f"dbfs:/student-groups/Group_3_1/mvp/random_forest/checkpoints/testnonLeakage"))

In [0]:
last_cv_model = RandomForestClassifier.load(f"dbfs:/student-groups/Group_3_1/mvp/random_forest/checkpoints/testnonLeakage/rf_window_4")
last_cv_model

In [0]:
model = last_cv_model

In [0]:

# Save results
results_df = spark.createDataFrame(pd.DataFrame(results))
results_df.write.mode("overwrite").parquet(f"{result_dir}/cv_results.parquet")
display(results_df)

# Final model on all 2015-2017, test on 2019
# train_all = spark.read.parquet(f"{base_dir}/train")

test_2019 = spark.read.parquet(f"{base_dir}/test/YEAR=2019")
test_pred = model.transform(test_2019)
tp_test, fp_test, fn_test, tn_test, precision_test, recall_test, f2_test = f2_score(test_pred)
pr_auc_test = pr_evaluator.evaluate(test_pred)

print(f"tp_test: {tp_test}")
print(f"fp_test: {fp_test}")
print(f"fn_test: {fn_test}")
print(f"tn_test: {tn_test}")
print(f"precision_test: {precision_test}")
print(f"recall_test: {recall_test}")
print(f"f2_test: {f2_test}")
print(f"pr_auc_test: {pr_auc_test}")

test_results = spark.createDataFrame([{
    "tp_test": tp_test,
    "fp_test": fp_test,
    "fn_test": fn_test,
    "precision_test": precision_test,
    "recall_test": recall_test,
    "f2_test": f2_test,
    "pr_auc_test": pr_auc_test
}])
test_results.write.mode("overwrite").parquet(f"{result_dir}/test_results.parquet")
display(test_results)

In [0]:
# pull out test_results 
test_results = spark.read.parquet(f"{result_dir}/test_results.parquet")
test_results = test_results.toPandas()


In [0]:
# Display test confusion matrix from the best model
test_confusion_matrix = np.array([[tn_test, fp_test],
                                    [fn_test, tp_test]])

In [0]:
fig, axes = plt.subplots(4, 2, figsize=(18, 28))
ax1 = axes[0, 0]
im = ax1.imshow(test_confusion_matrix, cmap="Blues", aspect="auto")
ax1.set_xticks([0, 1])
ax1.set_yticks([0, 1])
ax1.set_xticklabels(["Predicted 0\n(Not Delayed)", "Predicted 1\n(Delayed)"], fontsize=11)
ax1.set_yticklabels(["Actual 0\n(Not Delayed)", "Actual 1\n(Delayed)"], fontsize=11)
ax1.set_title(f"Test Dataset Confusion Matrix", fontsize=13, fontweight="bold")
for i in range(2):
    for j in range(2):
        val = test_confusion_matrix[i, j]
        color = "white" if val > test_confusion_matrix.max() / 2 else "black"
        label_text = ["TN", "FP", "FN", "TP"][i * 2 + j]
        ax1.text(j, i, f"{label_text}\n{val:,}", ha="center", va="center", fontsize=13, color=color, fontweight="bold")
fig.colorbar(im, ax=ax1, shrink=0.8)


# Grid Search

In [0]:
# creating a grid search for Random Forest Hyperparameter Tuning

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

train_df = spark.read.parquet(f"{base_dir}/train")
test_df = spark.read.parquet(f"{base_dir}/test")

# Define the parameter grid
param_grid = (ParamGridBuilder()
              .addGrid(rf.numTrees, [50, 100])
              .addGrid(rf.maxDepth, [4, 8, 16])
              .build())

print("Defined parameter grid: ", param_grid)
# Define the evaluation metric
evaluator = pr_evaluator

# Define the cross-validator
cv = CrossValidator(estimator=rf, evaluator=evaluator, estimatorParamMaps=param_grid, numFolds=3)

print("Starting cross-validation")

# Train the model using the cross-validator
cv_model = cv.fit(train_df)
# Save the best model
best_model = cv_model.bestModel
best_model.write().overwrite().save(f"{checkpoint_dir}/testGridSearch/bestModel")

print(f"Wrote best model to {checkpoint_dir}/testGridSearch/bestModel")

# Load the best model
best_model = RandomForestClassifier.load(f"{checkpoint_dir}/testGridSearch/bestModel")

print("Starting train evaluation")

# Evaluate the best model on the train set
train_pred = best_model.transform(train_df)
tp_train, fp_train, fn_train, precision_train, recall_train, f2_train = f2_score(train_pred)
pr_auc_train = pr_evaluator.evaluate(train_pred)
print(f"tp_train: {tp_train}")
print(f"fp_train: {fp_train}")
print(f"fn_train: {fn_train}")
print(f"precision_train: {precision_train}")
print(f"recall_train: {recall_train}")
print(f"f2_train: {f2_train}")
print(f"pr_auc_train: {pr_auc_train}")

train_results = spark.createDataFrame([{
    "tp_train": tp_train,
    "fp_train": fp_train,
    "fn_train": fn_train,
    "precision_train": precision_train,
    "recall_train": recall_train,
    "f2_train": f2_train,
    "pr_auc_train": pr_auc_train
}])
train_results.write.mode("overwrite").parquet(f"{grid_search_results_dir}/train_results50-100_4-8-16.parquet")
display(train_results)

print("Starting test evaluation")

# Evaluate the best model on the test set
test_pred = best_model.transform(test_df)
tp_test, fp_test, fn_test, precision_test, recall_test, f2_test = f2_score(test_pred)
pr_auc_test = pr_evaluator.evaluate(test_pred)
print(f"tp_test: {tp_test}")
print(f"fp_test: {fp_test}")
print(f"fn_test: {fn_test}")
print(f"precision_test: {precision_test}")
print(f"recall_test: {recall_test}")
print(f"f2_test: {f2_test}")
print(f"pr_auc_test: {pr_auc_test}")

test_results = spark.createDataFrame([{
    "tp_test": tp_test,
    "fp_test": fp_test,
    "fn_test": fn_test,
    "precision_test": precision_test,
    "recall_test": recall_test,
    "f2_test": f2_test,
    "pr_auc_test": pr_auc_test
}])
test_results.write.mode("overwrite").parquet(f"{grid_search_results_dir}/test_results50-100_4-8-16.parquet")
display(test_results)

In [0]:
grid_search_model = best_model

In [0]:
# Get depth of each tree
tree_depths = [tree.depth for tree in best_model.trees]

# Calculate statistics
print(f"Number of trees: {len(tree_depths)}")
print(f"Maximum depth: {max(tree_depths)}")

In [0]:
model_path = f"{checkpoint_dir}/testGridSearch/bestModel"
grid_search_model.write().overwrite().save(model_path)

In [0]:
df_test.count()

In [0]:
print(df_test.count())
# Using the results from the original Random Forest:
tp_test = 1348469
fp_test = 2629259
fn_test = 7817
tn_test = df_test.count() - (tp_train + fp_train + fn_train)

In [0]:
tn_test